# 12 — Capstone: ML Platform

This capstone integrates all MLOps layers into a single end-to-end platform walk-through: experiment tracking → model registry → CI/CD gate → A/B test → monitoring → drift-triggered retraining. Each component is a simplified version of those built in earlier notebooks, composed into a working pipeline.

In [ ]:
# ML Platform Capstone — Full Pipeline
import numpy as np
import json, hashlib, time
from collections import deque
from scipy import stats

print('='*60)
print('ML Platform Capstone: Credit Risk Scoring Pipeline')
print('='*60)

# === 1. EXPERIMENT TRACKING ===
print('\n[1] EXPERIMENT TRACKING')
np.random.seed(42)
runs = []
for i, (lr, depth) in enumerate([(0.01,4),(0.05,6),(0.001,3)]):
    auc = 0.80 + depth*0.01 - lr*0.5 + np.random.normal(0, 0.005)
    run = {
        'id': hashlib.md5(f'run{i}'.encode()).hexdigest()[:8],
        'params': {'lr':lr, 'depth':depth},
        'metrics': {'val_auc': round(auc,4), 'val_precision':round(auc-0.03,4)}
    }
    runs.append(run)
    print(f'  Run {run["id"]}: lr={lr} depth={depth} -> val_auc={auc:.4f}')
best_run = max(runs, key=lambda r: r['metrics']['val_auc'])
print(f'  Best run: {best_run["id"]} (auc={best_run["metrics"]["val_auc"]})')

In [ ]:
# === 2. MODEL REGISTRY ===
print('\n[2] MODEL REGISTRY')
registry = {
    'credit_risk': [
        {'version': 1, 'run_id': best_run['id'], 'metrics': best_run['metrics'], 'stage': 'None'}
    ]
}
registry['credit_risk'][0]['stage'] = 'Staging'
print(f'  Registered v1 from run {best_run["id"]} -> Staging')

# === 3. CI/CD GATE ===
print('\n[3] CI/CD ACCEPTANCE GATE')
gates = {
    'val_auc':      ('min', 0.80),
    'val_precision':('min', 0.75),
}
failures = []
for metric, (direction, threshold) in gates.items():
    val = best_run['metrics'][metric]
    if direction == 'min' and val < threshold:
        failures.append(f'{metric}={val} < {threshold}')
if failures:
    print(f'  CI FAILED: {failures}')
else:
    registry['credit_risk'][0]['stage'] = 'Production'
    print('  CI PASSED -> Promoted to Production')

# === 4. A/B TEST ===
print('\n[4] A/B TEST (10% traffic to challenger)')
np.random.seed(0)
ctrl_outcomes = np.random.binomial(1, 0.74, 900)    # champion
trt_outcomes  = np.random.binomial(1, 0.79, 100)    # challenger
t, p = stats.ttest_ind(ctrl_outcomes, trt_outcomes)
print(f'  Champion mean={ctrl_outcomes.mean():.3f}  Challenger mean={trt_outcomes.mean():.3f}')
print(f'  t={t:.3f} p={p:.4f} -> {"PROMOTE challenger" if p<0.05 and trt_outcomes.mean()>ctrl_outcomes.mean() else "Keep champion"}')

In [ ]:
# === 5. PRODUCTION MONITORING ===
print('\n[5] PRODUCTION MONITORING')
latencies = np.random.gamma(2, 25, 1000)
pred_scores = np.random.beta(2, 8, 1000)
print(f'  Latency p50={np.percentile(latencies,50):.1f}ms p99={np.percentile(latencies,99):.1f}ms')
print(f'  Pred mean={pred_scores.mean():.4f} std={pred_scores.std():.4f}')
print('  Status: OK')

# === 6. DRIFT DETECTION & RETRAINING ===
print('\n[6] DRIFT DETECTION & RETRAINING TRIGGER')
reference = np.random.exponential(100, 5000)

def psi_simple(ref, act, n_bins=10):
    bins = np.percentile(ref, np.linspace(0,100,n_bins+1))
    bins[0], bins[-1] = -np.inf, np.inf
    e = np.clip(np.histogram(ref, bins)[0]/len(ref), 1e-6, None)
    a = np.clip(np.histogram(act, bins)[0]/len(act), 1e-6, None)
    return float(np.sum((a-e)*np.log(a/e)))

model_version = 1
for month, scale in [(1,105),(2,115),(3,140),(4,200)]:
    incoming = np.random.exponential(scale, 1000)
    p = psi_simple(reference, incoming)
    if p > 0.25:
        model_version += 1
        reference = incoming
        status = f'DRIFT (PSI={p:.3f}) -> retrained -> v{model_version}'
    else:
        status = f'OK (PSI={p:.3f})'
    print(f'  Month {month}: {status}')

# === SUMMARY ===
print('\n' + '='*60)
print('PLATFORM SUMMARY')
print('='*60)
print(f'  Experiments run:     {len(runs)}')
print(f'  Model promoted:      v1 -> Production')
print(f'  CI gates passed:     Yes')
print(f'  A/B test result:     Challenger evaluated')
print(f'  Monitoring:          Active (latency + prediction drift)')
print(f'  Auto-retrains:       {model_version-1}')
print(f'  Current model:       v{model_version}')